In [1]:
import torch
import torch.nn as nn
import torch.optim as optim #导入优化器
import numpy as np
import akshare as ak
import pandas as pd
from tqdm import tqdm   # 导入tqdm库，显示进度条
import matplotlib.pyplot as plt
from copy import deepcopy as copy
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# 获取数据
class GetData:
    def __init__(self,stock_id,save_path,start="20100101", end=None):
        """初始化方法
        :param stock_id: 股票代码
        :param save_path: 数据保存路径
        """
        self.stock_id=stock_id
        self.save_path=save_path
        self.start=start
        self.end=end or pd.Timestamp.today().strftime("%Y%m%d")
        self.data=None
    
    def getData(self):
        """获取数据
        将数据进行处理后，保存成文件
        """
        # 获取历史数据，并按时间顺序排序
        df = ak.stock_zh_a_daily(symbol=self.stock_id,
                                start_date=self.start,
                                end_date=self.end)
        df = df[['open','close','high','low','volume']]   # 若是 vol 请改成 df['vol']
        df = df.sort_index()      # 按日期升序
        self.data=df.copy()
        # 计算数据列的最大值、最小值
        self.close_max=self.data['close'].max()
        self.close_min=self.data['close'].min()
        # 归一化
        self.data=self.data.apply(lambda x:(x-min(x))/(max(x)-min(x)))
        # 保存
        self.data.to_csv(self.save_path)
        
        return self.data
    
    def process_data(self,n):
        """处理数据
        区分特征和标签，划分训练集和验证集
        :param n: 滑动窗口大小
        :return: 训练集和验证集的特征和标签
        """
        if self.data is None:
            print("请先获取数据")
            self.getData()
        # 提取特征和标签
        feature=[
            self.data.iloc[i:i+n].values.tolist() 
            for i in range(len(self.data)-n+2)
            if i+n<len(self.data)
        ]
        label=[
            self.data.close.values[i+n]
            for i in range(len(self.data)-n+2)
            if i+n<len(self.data)
        ]
        # 划分训练集和验证集
        train_X=feature[:500]
        test_X=feature[500:]
        train_y=label[:500]
        test_y=label[500:]
        
        return train_X,train_y,test_X,test_y

***LSTM模型，长短期记忆网络***

In [3]:
class Model(nn.Module):# 继承
    def __init__(self,n): 
        super(Model,self).__init__()
        # 创建一个LSTM层，输入大小为n，隐藏大小为256，批次优先为True
        self.lstm_layer=nn.LSTM(input_size=n,hidden_size=256,batch_first=True)
        # 创建一个线性层，输入大小为256，输出大小为1，有偏差
        self.linear_layer=nn.Linear(in_features=256,out_features=1,bias=True)
    
    def forward(self,x):
        # 通过LSTM层处理x，得到输出out1和隐藏状态h_n、h_c
        out1,(h_n,h_c)=self.lstm_layer(x)
        a,b,c=h_n.shape # 获取h_n的形状，分别赋值给a、b、c
        # 将h_c重塑为(a*b,c)的形状，通过线性层处理，得到out2
        out2=self.linear_layer(h_n.reshape(a*b,c))
        return out2

***模型训练***

In [8]:
def train_model(epoch,train_dataLoader,test_dataLoader):
    """
    参数：
    - epoch: 训练的轮数
    - train_dataLoader: 训练数据的DataLoader对象
    - test_dataLoader: 测试数据的DataLoader对象
    """
    best_model=None # 初始化最佳模型为None
    train_loss=0
    test_loss=0
    best_loss=100
    epoch_cnt=0
    
    for _ in range(epoch):
        total_train_loss=0
        total_train_num=0
        total_test_loss=0
        total_test_num=0
        
        for x,y in tqdm(train_dataLoader,desc='Epoch:{}|Train Loss: {} | Test Loss: {}'
                        .format(_,train_loss,test_loss)):
            x_num=len(x) # 获取当前批次的样本数量
            p=model(x)                         # 使用已有模型进行前向传播
            loss=loss_func(p,y) 
            optimizer.zero_grad() # 清空梯度
            loss.backward() # 反向传播计算梯度
            optimizer.step() # 更新模型参数
            total_train_loss+=loss.item()
            total_train_num+=x_num
        train_loss=total_train_loss/total_train_num # 计算平均训练损失
        
        for x,y in test_dataLoader:
            x_num=len(x)
            p=model(x)
            loss=loss_func(p,y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_test_loss+=loss.item()
            total_test_num+=x_num
        test_loss=total_test_loss/total_test_num # 计算平均测试损失
        if test_loss<best_loss: # 如果当前测试损失小于最佳损失
            best_loss=test_loss # 更新最佳损失
            best_model=copy(model) # 复制当前模型为最佳模型
            epoch_cnt=0
        else:
            epoch_cnt+=1
        
        if epoch_cnt>early_stop:
            torch.save(best_model.state_dict(),'./lstm.pth') # 保存最佳模型的参数到文件
            break
            

In [ ]:
# 定义测试模型的函数
def test_model(test_dataLoader):
    """
    参数：
    - test_dataLoader: 测试数据的DataLoader对象
    """
    pred=[]
    label=[]
    model_=Model(5) # 创建一个Model对象，输入大小为5
    model_.load_state_dict(torch.load('./lstm.pth')) # 加载保存的模型参数
    model_.eval() # 设置模型为评估模式
    total_test_loss=0
    total_test_num=0
    
    for x,y in test_dataLoader:
        x_num=len(x)
        p=model_(x) # 使用模型进行预测
        loss=nn.MSELoss()(p,y) # 计算均方误差损失
        total_test_loss+=loss.item() # 累加测试损失
        total_test_num+=x_num # 累加测试样本数量
        pred.extend(p.data.squeeze(1).tolist()) 
        label.extend(y.tolist())
    
    test_loss=total_test_loss/total_test_num # 计算平均测试损失
    return pred,label,test_loss


In [6]:
def plot_img(data,pred):
    plt.rcParams['font.sans-serif']=['SimHei'] # 设置中文字体为SimHei
    plt.figure(figsize=(12,7)) # 创建一个10x5英寸的图形
    plt.plot(range(len(pred),pred,color='red',label='预测值')) # 绘制预测值的折线图，颜色为红色，标签为'预测值'
    plt.plot(range(len(data)),data,color='blue',label='真实值') #
    
    for i in range(0,len(pred)-3,5):
        price=[data[i]+pred[j]-pred[i] for j in range(i,i+3)]
        plt.plot(range(i,i+3),price,color='red')
        
    plt.xticks(fontproperties='Times New Roman',size=14) # 设置x轴刻度的字体和大小
    plt.yticks(fontproperties='Times New Roman',size=14) # 设置y轴
    plt.xlabel('日期',fontsize=18)
    plt.ylabel('成交量',fontsize=18)
    plt.title('股票成交量趋势预测',fontsize=20)
    plt.show()
    

In [7]:
# 超参数设置
days_num=5 # 滑动窗口大小
epoch=20
fea=5
batch_size=20
early_stop=5

model=Model(fea) # 创建一个Model对象，输入大小为fea

GD=GetData(stock_id='sh601398',save_path='./data.csv') # 创建一个GetData对象，股票代码为'601398'，数据保存路径为'./data.csv'
train_X,train_y,test_X,test_y=GD.process_data(days_num) # 处理数据，获取训练集和测试集的特征和标签
train_X=torch.tensor(train_X,dtype=torch.float32) # 将训练特征转换为PyTorch张量，数据类型为float32
train_y=torch.tensor(train_y,dtype=torch.float32).unsqueeze(1) 
test_X=torch.tensor(test_X,dtype=torch.float32) # 将测试特征转换为PyTorch张量，数据类型为float32
test_y=torch.tensor(test_y,dtype=torch.float32).unsqueeze(1)

train_data=TensorDataset(train_X,train_y) # 创建一个TensorDataset对象，包含训练特征和标签
test_data=TensorDataset(test_X,test_y) 
train_dataLoader=DataLoader(train_data,batch_size=batch_size) 
test_dataLoader=DataLoader(test_data,batch_size=batch_size)

loss_func=nn.MSELoss() # 定义损失函数为均方误差
optimizer=optim.Adam(model.parameters(),lr=0.001) # 定义优化器为Adam，学习率为0.001

# 训练模型
train_model(epoch,train_dataLoader,test_dataLoader)
p,y,test_loss=test_model(test_dataLoader) # 测试模型，获取预测值、真实值和测试损失

# 处理预测值和真实值
pred=[ele * (GD.close_max-GD.close_min)+GD.close_min for ele in p] # 将预测值反归一化
data=[ele * (GD.close_max-GD.close_min)+GD.close_min for ele in y] # 将真实值反归一化

# 绘图
plot_img(data,pred)

# 输出测试损失
print("测试损失：",test_loss)

请先获取数据


Epoch:0|Train Loss: 0 | Test Loss: 0:   0%|          | 0/25 [00:00<?, ?it/s]


TypeError: empty(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got Tensor"